# Module 5: Deep Reinforcement Learning
## Notebook 1: DQN from Scratch

**Learning Objectives**

By completing this notebook, you will:
1. Build a complete Deep Q-Network (DQN) from scratch using PyTorch
2. Implement a replay buffer and target network and understand why both are necessary
3. Train DQN on CartPole-v1 until it solves the environment
4. Run an ablation study showing performance collapse when either component is removed

**Prerequisites**
- Q-learning (Module 3)
- Linear function approximation (Module 4)
- Basic PyTorch (nn.Module, Adam optimizer, backpropagation)

**Estimated Time: 15 minutes**

---

### Why DQN is Not Just "Deep Q-Learning"

Naively replacing the Q-table with a neural network fails. Two instabilities arise:

1. **Correlated samples**: consecutive (s, a, r, s') transitions are highly correlated. Gradient descent on correlated data causes the network to forget previous experience and oscillate.

2. **Moving targets**: the same network $Q_\theta$ generates both the prediction and the target ($r + \gamma \max_{a'} Q_\theta(s', a')$). As weights update, the target moves too — like chasing a moving goalpost.

DQN's two fixes:
- **Replay buffer**: store transitions, sample random mini-batches (breaks correlation)
- **Target network**: a periodic copy of the main network used only for computing targets (stabilizes the goalposts)

In [ ]:
learning_objectives([
    "By completing this notebook, you will:",
    "- Q-learning (Module 3)",
    "Linear function approximation (Module 4)",
    "Basic PyTorch (nn.Module, Adam optimizer, backpropagation)",
    "---",
    "**Replay buffer**: store transitions, sample random mini-batches (breaks correlation)",
    "**Target network**: a periodic copy of the main network used only for computing targets (stabilizes the goalposts)"
])

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
apply_course_theme()
apply_plot_theme()

In [ ]:
# Purpose: Import all dependencies and set reproducibility seeds
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import gymnasium as gym
import matplotlib.pyplot as plt
from collections import deque
import random

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)

# Use CPU — CartPole is fast enough
DEVICE = torch.device('cpu')

plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {DEVICE}")

# Quick env check
env_test = gym.make('CartPole-v1')
print(f"CartPole-v1: obs_dim={env_test.observation_space.shape[0]}, "
      f"n_actions={env_test.action_space.n}")
env_test.close()

## 1. Replay Buffer

The replay buffer stores transitions $(s, a, r, s', \text{done})$ and provides random mini-batch sampling. We use a circular buffer (deque with maxlen) so old experiences are automatically discarded.

In [ ]:
# Purpose: Implement the experience replay buffer
# Key Concept: Random sampling from the buffer breaks temporal correlation in training

class ReplayBuffer:
    """
    Fixed-size circular buffer storing (state, action, reward, next_state, done) tuples.

    Parameters
    ----------
    capacity : int — maximum number of transitions to store
    """

    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        """
        Store a single transition.

        Parameters
        ----------
        state      : np.ndarray
        action     : int
        reward     : float
        next_state : np.ndarray
        done       : bool
        """
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        """
        Sample a random mini-batch.

        Returns
        -------
        Tuple of batched tensors: (states, actions, rewards, next_states, dones)
        Each tensor is on DEVICE.
        """
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)

        return (
            torch.tensor(np.array(states),      dtype=torch.float32).to(DEVICE),
            torch.tensor(actions,               dtype=torch.long).to(DEVICE),
            torch.tensor(rewards,               dtype=torch.float32).to(DEVICE),
            torch.tensor(np.array(next_states), dtype=torch.float32).to(DEVICE),
            torch.tensor(dones,                 dtype=torch.float32).to(DEVICE),
        )

    def __len__(self):
        return len(self.buffer)


# Sanity check
buf = ReplayBuffer(capacity=1000)
for _ in range(50):
    buf.push(np.zeros(4), 0, 1.0, np.zeros(4), False)

states, actions, rewards, next_states, dones = buf.sample(batch_size=16)
print(f"Buffer size: {len(buf)}")
print(f"Sample batch — states: {states.shape}, actions: {actions.shape}, rewards: {rewards.shape}")

## 2. Q-Network Architecture

We use a simple 3-layer MLP: obs_dim → 128 → 128 → n_actions. The output is Q-values for all actions simultaneously — this is more efficient than calling the network once per action.

In [ ]:
# Purpose: Define the Q-network neural network architecture
# Key Concept: Output layer has n_actions units, one Q-value per action

class QNetwork(nn.Module):
    """
    3-layer MLP mapping observations to Q-values.

    Input:  observation vector (batch_size, obs_dim)
    Output: Q-values for all actions (batch_size, n_actions)
    """

    def __init__(self, obs_dim, n_actions, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, n_actions),
        )

    def forward(self, x):
        """
        Parameters
        ----------
        x : torch.Tensor, shape (batch_size, obs_dim)

        Returns
        -------
        q_values : torch.Tensor, shape (batch_size, n_actions)
        """
        return self.net(x)


# CartPole-v1 dimensions
OBS_DIM   = 4   # [cart_pos, cart_vel, pole_angle, pole_vel]
N_ACTIONS = 2   # [push_left, push_right]

q_net = QNetwork(OBS_DIM, N_ACTIONS).to(DEVICE)
print(q_net)
n_params = sum(p.numel() for p in q_net.parameters())
print(f"Total parameters: {n_params:,}")

# Test forward pass
test_obs = torch.zeros(1, OBS_DIM)
q_vals   = q_net(test_obs)
print(f"Q-values for zero state: {q_vals.detach().numpy()}")

## 3. The DQN Agent

The agent encapsulates:
- The online Q-network (updated every step via gradient descent)
- The target Q-network (copied from online every `target_update_freq` steps)
- The replay buffer
- Epsilon-greedy action selection

In [ ]:
# Purpose: Implement the complete DQN agent
# Key Concept: Online network is trained; target network provides stable TD targets

class DQNAgent:
    """
    Deep Q-Network agent with experience replay and target network.

    Parameters
    ----------
    obs_dim           : int
    n_actions         : int
    lr                : float — Adam learning rate
    gamma             : float — discount factor
    epsilon_start     : float — initial exploration rate
    epsilon_end       : float — minimum exploration rate
    epsilon_decay     : float — multiplicative decay per step
    buffer_capacity   : int — replay buffer size
    batch_size        : int — mini-batch size for each gradient step
    target_update_freq: int — steps between target network updates
    use_replay        : bool — ablation: disable replay buffer
    use_target        : bool — ablation: disable target network
    """

    def __init__(self, obs_dim, n_actions, lr=1e-3, gamma=0.99,
                 epsilon_start=1.0, epsilon_end=0.01, epsilon_decay=0.995,
                 buffer_capacity=10_000, batch_size=64, target_update_freq=100,
                 use_replay=True, use_target=True):

        self.n_actions    = n_actions
        self.gamma        = gamma
        self.epsilon      = epsilon_start
        self.epsilon_end  = epsilon_end
        self.eps_decay    = epsilon_decay
        self.batch_size   = batch_size
        self.target_freq  = target_update_freq
        self.use_replay   = use_replay
        self.use_target   = use_target
        self.steps        = 0

        # Online Q-network: trained every step
        self.q_net    = QNetwork(obs_dim, n_actions).to(DEVICE)

        # Target Q-network: periodic copy, not directly trained
        self.q_target = QNetwork(obs_dim, n_actions).to(DEVICE)
        self.q_target.load_state_dict(self.q_net.state_dict())
        self.q_target.eval()   # target network is always in eval mode

        self.optimizer = optim.Adam(self.q_net.parameters(), lr=lr)
        self.buffer    = ReplayBuffer(buffer_capacity)

    def select_action(self, state):
        """
        Epsilon-greedy action selection.

        Returns
        -------
        int — selected action
        """
        if random.random() < self.epsilon:
            return random.randrange(self.n_actions)   # explore

        with torch.no_grad():
            obs_t = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(DEVICE)
            return int(self.q_net(obs_t).argmax(dim=1).item())  # exploit

    def update(self, state, action, reward, next_state, done):
        """
        Store transition and perform one gradient update.

        Returns
        -------
        float or None — training loss (None if buffer not ready)
        """
        # Step 1: Store transition in replay buffer
        self.buffer.push(state, action, reward, next_state, done)
        self.steps += 1

        # Step 2: Decay epsilon
        self.epsilon = max(self.epsilon_end, self.epsilon * self.eps_decay)

        # Step 3: Don't train until buffer has enough transitions
        if len(self.buffer) < self.batch_size:
            return None

        # Step 4: Sample mini-batch from buffer (or use last transition)
        if self.use_replay:
            states, actions, rewards, next_states, dones = self.buffer.sample(self.batch_size)
        else:
            # Ablation: train on the single last transition (no replay)
            states     = torch.tensor(state,      dtype=torch.float32).unsqueeze(0).to(DEVICE)
            actions    = torch.tensor([action],   dtype=torch.long).to(DEVICE)
            rewards    = torch.tensor([reward],   dtype=torch.float32).to(DEVICE)
            next_states= torch.tensor(next_state, dtype=torch.float32).unsqueeze(0).to(DEVICE)
            dones      = torch.tensor([float(done)], dtype=torch.float32).to(DEVICE)

        # Step 5: Compute current Q-values for selected actions
        q_values = self.q_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)

        # Step 6: Compute target Q-values
        with torch.no_grad():
            net_for_target = self.q_target if self.use_target else self.q_net
            max_next_q = net_for_target(next_states).max(dim=1).values
            targets    = rewards + self.gamma * max_next_q * (1.0 - dones)

        # Step 7: Compute MSE loss and backpropagate
        loss = F.mse_loss(q_values, targets)
        self.optimizer.zero_grad()
        loss.backward()
        # Gradient clipping for stability
        torch.nn.utils.clip_grad_norm_(self.q_net.parameters(), max_norm=10.0)
        self.optimizer.step()

        # Step 8: Periodically sync target network
        if self.use_target and self.steps % self.target_freq == 0:
            self.q_target.load_state_dict(self.q_net.state_dict())

        return loss.item()


print("DQNAgent defined.")

# Quick sanity check
agent_test = DQNAgent(OBS_DIM, N_ACTIONS)
dummy_obs  = np.zeros(OBS_DIM)
a = agent_test.select_action(dummy_obs)
print(f"Test action: {a} (epsilon={agent_test.epsilon:.2f})")

## 4. Training Loop

We train DQN on CartPole-v1 for up to 300 episodes. CartPole is considered "solved" when the mean episode reward over 100 consecutive episodes is >= 475.

In [ ]:
# Purpose: Implement the DQN training loop
# Key Concept: Episode rewards track task performance; epsilon tracks exploration decay

def train_dqn(agent_kwargs, n_episodes=300, seed=42, verbose=True):
    """
    Train a DQN agent on CartPole-v1.

    Parameters
    ----------
    agent_kwargs : dict — kwargs for DQNAgent (allows ablation configurations)
    n_episodes   : int
    seed         : int

    Returns
    -------
    episode_rewards : list of float
    epsilons        : list of float — epsilon at end of each episode
    """
    env   = gym.make('CartPole-v1')
    agent = DQNAgent(OBS_DIM, N_ACTIONS, **agent_kwargs)

    episode_rewards = []
    epsilons        = []

    for ep in range(n_episodes):
        obs, _ = env.reset(seed=seed + ep)
        total_reward = 0.0
        done = False

        while not done:
            action = agent.select_action(obs)
            next_obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            agent.update(obs, action, reward, next_obs, done)

            total_reward += reward
            obs = next_obs

        episode_rewards.append(total_reward)
        epsilons.append(agent.epsilon)

        if verbose and (ep + 1) % 50 == 0:
            mean_100 = np.mean(episode_rewards[-100:])
            print(f"Episode {ep+1:4d}: mean-100={mean_100:.1f}, epsilon={agent.epsilon:.3f}")

    env.close()
    return episode_rewards, epsilons


# Full DQN (replay + target network)
print("=== Training Full DQN ===")
FULL_DQN_KWARGS = dict(
    lr=1e-3, gamma=0.99,
    epsilon_start=1.0, epsilon_end=0.01, epsilon_decay=0.995,
    buffer_capacity=10_000, batch_size=64, target_update_freq=100,
    use_replay=True, use_target=True,
)
rewards_full, eps_full = train_dqn(FULL_DQN_KWARGS, n_episodes=300)
print(f"Final mean-100 reward: {np.mean(rewards_full[-100:]):.1f}")

## 5. Ablation Study: What Happens Without Replay or Target Network?

In [ ]:
# Purpose: Ablation — show DQN without replay buffer collapses
# Key Concept: Without replay, training on correlated sequential transitions causes instability

print("=== Training DQN without Replay Buffer ===")
NO_REPLAY_KWARGS = dict(
    lr=1e-3, gamma=0.99,
    epsilon_start=1.0, epsilon_end=0.01, epsilon_decay=0.995,
    buffer_capacity=10_000, batch_size=64, target_update_freq=100,
    use_replay=False, use_target=True,   # ← only change
)
rewards_no_replay, _ = train_dqn(NO_REPLAY_KWARGS, n_episodes=300)
print(f"Final mean-100 reward: {np.mean(rewards_no_replay[-100:]):.1f}")

In [ ]:
# Purpose: Ablation — show DQN without target network collapses
# Key Concept: Without target network, the optimization target moves every step

print("=== Training DQN without Target Network ===")
NO_TARGET_KWARGS = dict(
    lr=1e-3, gamma=0.99,
    epsilon_start=1.0, epsilon_end=0.01, epsilon_decay=0.995,
    buffer_capacity=10_000, batch_size=64, target_update_freq=100,
    use_replay=True, use_target=False,   # ← only change
)
rewards_no_target, _ = train_dqn(NO_TARGET_KWARGS, n_episodes=300)
print(f"Final mean-100 reward: {np.mean(rewards_no_target[-100:]):.1f}")

In [ ]:
# Purpose: Plot all three learning curves together for comparison
# Key Concept: Ablation reveals each component's contribution to stability

def smooth(data, window=15):
    return np.convolve(data, np.ones(window)/window, mode='valid')


fig, ax = plt.subplots(figsize=(13, 6))

configs = [
    (rewards_full,       'Full DQN (replay + target)',   'steelblue'),
    (rewards_no_replay,  'No Replay Buffer',              'tomato'),
    (rewards_no_target,  'No Target Network',             'darkorange'),
]

for rewards, label, color in configs:
    ax.plot(rewards, color=color, alpha=0.25, linewidth=0.8)
    ax.plot(smooth(rewards), color=color, linewidth=2.5, label=label)

ax.axhline(475, color='green', linestyle='--', linewidth=1.5,
           label='Solved threshold (475)')
ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Episode Reward', fontsize=12)
ax.set_title('DQN Ablation Study on CartPole-v1', fontsize=14)
ax.legend(fontsize=11)
ax.set_ylim(-10, 510)
plt.tight_layout()
plt.savefig('../resources/dqn_ablation.png', dpi=120, bbox_inches='tight')
plt.show()

print("\nFinal performance summary (mean of last 100 episodes):")
for rewards, label, _ in configs:
    print(f"  {label:35s}: {np.mean(rewards[-100:]):.1f}")

### Exercise 5.1.1: Visualize Q-Values During Training

**Task:** Retrain the full DQN but record the mean max-Q-value at the start of each episode (before any training steps). Use the learned network to compute Q(obs, :) for the reset observation and take the max. Plot both episode rewards and mean max-Q over training.

**Expected Output:** Two curves on a shared x-axis (twin y-axes). The Q-value should generally increase alongside reward.

<details>
<summary>Hint 1</summary>
After env.reset(), call agent.q_net(torch.tensor(obs).unsqueeze(0)) with torch.no_grad(). Take .max().item() to get the max Q-value.
</details>

<details>
<summary>Hint 2</summary>
Use ax.twinx() to create a second y-axis for Q-values. This lets both curves share the episode x-axis while having independent scales.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

def train_dqn_with_qvals(agent_kwargs, n_episodes=300, seed=42):
    """
    Like train_dqn but also records max Q-value at episode start.

    Returns
    -------
    episode_rewards : list
    max_q_values    : list — max Q-value at episode start
    """
    env   = gym.make('CartPole-v1')
    agent = DQNAgent(OBS_DIM, N_ACTIONS, **agent_kwargs)

    episode_rewards = []
    max_q_values    = []

    for ep in range(n_episodes):
        obs, _ = env.reset(seed=seed + ep)

        # Record Q-value before episode steps
        with torch.no_grad():
            obs_t  = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(DEVICE)
            q_vals = agent.q_net(obs_t)
        max_q_values.append(q_vals.max().item())

        total_reward = 0.0
        done = False

        while not done:
            action = agent.select_action(obs)
            next_obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            agent.update(obs, action, reward, next_obs, done)
            total_reward += reward
            obs = next_obs

        episode_rewards.append(total_reward)

    env.close()
    return episode_rewards, max_q_values


rewards_qv, max_qs = train_dqn_with_qvals(FULL_DQN_KWARGS, n_episodes=300)

fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()

ax1.plot(smooth(rewards_qv), color='steelblue', linewidth=2, label='Episode Reward (smoothed)')
ax2.plot(smooth(max_qs),     color='darkorange', linewidth=2, alpha=0.8, label='Max Q-value (smoothed)')

ax1.set_xlabel('Episode', fontsize=12)
ax1.set_ylabel('Episode Reward', color='steelblue', fontsize=12)
ax2.set_ylabel('Max Q-value at Episode Start', color='darkorange', fontsize=12)
ax1.set_title('DQN: Episode Reward vs. Q-Value Growth', fontsize=13)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# AUTO-GRADED TESTS — Do not modify
# ----------------------------------
def test_exercise_511():
    assert len(max_qs) == 300, \
        f"max_q_values should have 300 entries (one per episode), got {len(max_qs)}"
    assert len(rewards_qv) == 300, \
        f"rewards should have 300 entries, got {len(rewards_qv)}"
    # Q-values should increase over training (late > early on average)
    early_q = np.mean(max_qs[:20])
    late_q  = np.mean(max_qs[-20:])
    assert late_q > early_q, (
        f"Q-values should increase during training. "
        f"Early mean={early_q:.2f}, late mean={late_q:.2f}"
    )
    print("Exercise 5.1.1 passed!")
    print(f"Q-value growth: {early_q:.2f} -> {late_q:.2f}")

test_exercise_511()

### Exercise 5.1.2: Target Network Update Frequency

**Task:** The `target_update_freq` hyperparameter controls how often the target network is synced. Train DQN with target_update_freq values [10, 50, 100, 500] for 250 episodes each (5 runs per config, seed 0-4). Plot the mean final performance vs. target_update_freq.

**Expected Output:** A bar chart showing performance across update frequencies. Very frequent updates (small freq) approach no-target performance; very infrequent updates (large freq) also hurt because the target becomes stale.

<details>
<summary>Hint 1</summary>
Use a loop: for freq in [10, 50, 100, 500]: run train_dqn with target_update_freq=freq, seed=0..4, and record np.mean(rewards[-50:]) for each run.
</details>

<details>
<summary>Hint 2</summary>
Use 5 runs per config (not 20) to keep total training time under 2 minutes.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

target_freqs   = [10, 50, 100, 500]
n_runs_per_cfg = 5
freq_means     = []
freq_stds      = []

for freq in target_freqs:
    run_finals = []
    kwargs = dict(
        lr=1e-3, gamma=0.99,
        epsilon_start=1.0, epsilon_end=0.01, epsilon_decay=0.995,
        buffer_capacity=10_000, batch_size=64,
        target_update_freq=freq,
        use_replay=True, use_target=True,
    )
    for run in range(n_runs_per_cfg):
        r, _ = train_dqn(kwargs, n_episodes=250, seed=run, verbose=False)
        run_finals.append(np.mean(r[-50:]))

    freq_means.append(np.mean(run_finals))
    freq_stds.append(np.std(run_finals))
    print(f"target_update_freq={freq:4d}: mean={freq_means[-1]:.1f} ± {freq_stds[-1]:.1f}")

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar([str(f) for f in target_freqs], freq_means,
       yerr=freq_stds, capsize=5, color='steelblue', edgecolor='navy')
ax.set_xlabel('Target Update Frequency (steps)', fontsize=12)
ax.set_ylabel('Mean Final Reward (last 50 eps, 5 runs)', fontsize=12)
ax.set_title('Effect of Target Network Update Frequency on DQN Performance', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# AUTO-GRADED TESTS — Do not modify
# ----------------------------------
def test_exercise_512():
    assert len(freq_means) == len(target_freqs), \
        f"Expected {len(target_freqs)} freq_means, got {len(freq_means)}"
    assert all(m >= 0 for m in freq_means), \
        "All mean rewards should be non-negative"
    # freq=100 should generally be a good default
    idx_100 = target_freqs.index(100)
    assert freq_means[idx_100] > 50, (
        f"freq=100 should achieve mean reward > 50, got {freq_means[idx_100]:.1f}. "
        "Check your implementation."
    )
    print("Exercise 5.1.2 passed!")
    best_freq = target_freqs[int(np.argmax(freq_means))]
    print(f"Best target_update_freq: {best_freq} (mean={max(freq_means):.1f})")

test_exercise_512()

In [ ]:
section_divider("Key Takeaways")

## Key Takeaways

1. **Replay buffer breaks correlation**: training on random mini-batches from a large buffer ensures the gradient estimates are statistically independent, preventing oscillation and catastrophic forgetting.
2. **Target network stabilizes training**: using a periodically-synced copy of the network for computing targets makes the optimization landscape more stationary. Without it, the loss landscape shifts every gradient step.
3. **Both components are necessary**: removing either one causes significant performance degradation or outright instability on CartPole. On harder environments (Atari), the effect is even more dramatic.
4. **Gradient clipping** is an important practical addition: it prevents exploding gradients in the early training phase when Q-value estimates are far from correct.
5. **Target update frequency** is a hyperparameter: too frequent (freq=10) approaches no-target behavior; too infrequent (freq=500) creates a stale target. freq=100-200 works well for CartPole.

## What's Next

Notebook 2 implements **Double DQN** — a targeted fix for Q-value overestimation bias that vanilla DQN exhibits. We will visualize how overestimation affects training and show that Double DQN produces more accurate Q-value estimates and better final performance.

In [ ]:
key_takeaways([
    "Review the key concepts covered in this notebook",
    "Practice applying these techniques to your own data",
    "Connect this material to the companion guide and slides"
])